# DwC-A to GeoPackage Converter

This script converts biodiversity data from Darwin Core Archive (DwC-A) format to GeoPackage format. It downloads the archive, extracts occurrence data, processes geometries, and saves the result as a spatial database file (GeoPackage).

## Step 1: Import Libraries

Import required packages for data handling, geospatial operations, and file management.

In [ ]:
import geopandas as gpd
import pandas as pd
import requests
import zipfile
import os
from shapely.geometry import MultiPoint, MultiLineString, MultiPolygon, GeometryCollection


## Step 2: Setup Configuration

Define the collection ID and paths for downloading and extracting the archive.

In [ ]:
# Main workflow
collection_id = "HR.1128"  # Change this to your collection ID

url = f"https://gbif.laji.fi/archive/{collection_id}"
zip_path = f"{collection_id}.zip"
extract_path = f"{collection_id}_extracted"

## Step 3: Download and Extract Data

Download the DwC-A archive from GBIF, extract it, locate the occurrence data file, and load it as a pandas DataFrame.

In [ ]:
# Download if not exists
if not os.path.exists(zip_path):
    print(f"Downloading archive from {url}...")
    response = requests.get(url)
    response.raise_for_status()
    with open(zip_path, 'wb') as f:
        f.write(response.content)
else:
    print(f"Using existing {zip_path}")

# Extract
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Find occurrence.txt
occurrences_file = None
for root, dirs, files in os.walk(extract_path):
    if 'occurrence.txt' in files:
        occurrences_file = os.path.join(root, 'occurrence.txt')
        break

if not occurrences_file:
    raise FileNotFoundError("occurrence.txt not found in archive")

# Read data
df = pd.read_csv(occurrences_file, sep='\t', low_memory=False)

## Step 4: Create GeoDataFrame

Convert WKT geometry strings from the footprintWKT column into spatial geometries and create a GeoDataFrame with EPSG:4326 (WGS84) CRS.

In [ ]:

# Create geometries
if 'footprintWKT' in df.columns and df['footprintWKT'].notna().any():
    geometry = gpd.GeoSeries.from_wkt(df['footprintWKT'])
else:
    raise ValueError("No valid geometry information found in the dataset. Please ensure that 'footprintWKT' column is present.")

gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")


## Step 5: Define Geometry Processing Functions

Helper functions to convert mixed geometry types into compatible formats (MultiPoint, MultiLineString, or MultiPolygon). After that conversion, the file can be opened in QGIS, for example.

In [ ]:
def convert_geometry_collection(geom):
    """Convert GeometryCollection to appropriate type."""
    if not isinstance(geom, GeometryCollection) or geom.is_empty:
        return geom

    geoms = list(geom.geoms)
    geom_types = set(g.geom_type for g in geoms)

    # Single geometry type
    if len(geom_types) == 1:
        geom_type = geom_types.pop()
        if geom_type == 'Point':
            return MultiPoint(geoms)
        elif geom_type == 'LineString':
            return MultiLineString(geoms)
        elif geom_type == 'Polygon':
            return MultiPolygon(geoms)

    # Mixed types - buffer and convert to MultiPolygon
    buffered = [g.buffer(0.5) for g in geoms]
    return MultiPolygon([b for b in buffered if b.geom_type == 'Polygon'])

In [ ]:
def process_geometries(gdf):
    """Convert GeometryCollections in GeoDataFrame."""
    if gdf.geometry.isnull().all():
        print("No geometries found in data")
        return gdf
    
    gdf['geometry'] = gdf.geometry.apply(convert_geometry_collection)
    return gdf

## Step 6: Process Geometries

Apply geometry conversion to handle any GeometryCollections, then display the types of geometries in the final dataset.

In [ ]:
# Process geometries
gdf = process_geometries(gdf)
print(f"Geometry types: {gdf.geometry.geom_type.unique()}")

## Step 7: Save to GeoPackage

Export the processed GeoDataFrame to a GeoPackage file for use in GIS applications.

In [ ]:
# Save to geopackage
output_file = f"{collection_id.replace('.', '_')}.gpkg"
gdf.to_file(output_file, driver='GPKG')
print(f"Saved to {output_file}")

Now the Darwin Core Archive is saved to a GeoPackage. Next, you can open it in QGIS or in other GIS application, and further process or analyze it in the way you want. 